    TF-IDF, keyword extraction, metadata features

In [2]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path("../").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.config import (
    CLEANED_TRAIN_PATH,
    CLEANED_TEST_PATH,
    FINAL_TRAIN_SVM_PATH,
    FINAL_TEST_SVM_PATH,
    TRAIN_LABEL_PATH
)

from src.utils.utils import load_csv, save_csv

from src.features.tfidf_features import TFIDFFeatureExtractor
from src.features.metadata_features import MetadataFeatureExtractor
from src.features.feature_union import FeatureUnion

print("✔ Imports loaded")

✔ Imports loaded


In [3]:
train_df = load_csv(CLEANED_TRAIN_PATH)
test_df = load_csv(CLEANED_TEST_PATH)

print(train_df.shape, test_df.shape)
train_df.head()

(2496, 8) (596, 7)


,id,title,venue,year,authors,doi,Label,abstract
0,0,proceedings 41st international conference on l...,iclp,2026,Paul Tarau,https://www.semanticscholar.org/paper/f7391104...,1,since the first conference in marseille in 198...
1,1,conditionals and temporal conditionals for gra...,iclp,2025,"Alexey Natekin, Alois Knoll",https://www.semanticscholar.org/paper/8328d53d...,1,gradient boosting machines are a family of pow...
2,2,learning and contesting assumption-based argum...,iclp,2025,"Ofer Arieli, Jesse Heyninck",https://www.semanticscholar.org/paper/1c433ff0...,2,our goal in this article is to remind readers ...
3,3,agentified argumentative learning short paper,iclp,2025,"Emanuele De Angelis, Maurizio Proietti, France...",https://www.semanticscholar.org/paper/666f0fa6...,1,argumentative learning amounts to integrating ...
4,4,empowering public interest communication with ...,iclp,2025,Alejandra Casas Niño de Rivera,https://www.semanticscholar.org/paper/08195948...,1,the epica empowering public interest communica...


In [4]:
tfidf = TFIDFFeatureExtractor(
    max_title_features=3000,
    max_abstract_features=8000,
    max_author_features=1500
)

metadata = MetadataFeatureExtractor()
union = FeatureUnion()

print("✔ Feature extractors initialized")

✔ Feature extractors initialized


In [5]:
X_train, train_meta_df = union.fit_transform(tfidf, metadata, train_df)

y_train = train_df["Label"].values

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (2496, 12508)
y_train shape: (2496,)


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

model = LinearSVC(C=1.0)
model.fit(X_tr, y_tr)

pred = model.predict(X_val)

print(classification_report(y_val, pred))

              precision    recall  f1-score   support

           1       0.50      0.57      0.53       181
           2       0.21      0.20      0.21       103
           3       0.18      0.15      0.16        88
           4       0.23      0.18      0.20        74
           5       0.47      0.54      0.50        54

    accuracy                           0.36       500
   macro avg       0.32      0.33      0.32       500
weighted avg       0.34      0.36      0.35       500



/Users/nhatnam/Documents/DM_252/Assignment/venv/lib/python3.10/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [7]:
X_test = union.transform(tfidf, metadata, test_df)

print("X_test shape:", X_test.shape)

X_test shape: (596, 12508)


In [9]:
from scipy import sparse
import os
import pandas as pd

# ensure output dir exists
os.makedirs(FINAL_TRAIN_SVM_PATH.parent, exist_ok=True)
os.makedirs(FINAL_TEST_SVM_PATH.parent, exist_ok=True)

# =========================
# SAVE SPARSE FEATURES
# =========================
train_npz_path = FINAL_TRAIN_SVM_PATH.with_suffix(".npz")
test_npz_path = FINAL_TEST_SVM_PATH.with_suffix(".npz")

sparse.save_npz(train_npz_path, X_train)
sparse.save_npz(test_npz_path, X_test)

# =========================
# SAVE LABELS (FIXED)
# =========================

pd.Series(y_train).to_csv(TRAIN_LABEL_PATH, index=False)

print("✔ Features saved to data/final/")
print("Train features:", train_npz_path)
print("Test features:", test_npz_path)
print("Labels:", TRAIN_LABEL_PATH)

# =========================
# PRINT SHAPES
# =========================
print("\n===== FEATURE SHAPES =====")
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\n===== LABEL SHAPES =====")
print("y_train length:", len(y_train))

# =========================
# PRINT FILE SIZES
# =========================
train_size_mb = os.path.getsize(train_npz_path) / (1024 * 1024)
test_size_mb = os.path.getsize(test_npz_path) / (1024 * 1024)
label_size_mb = os.path.getsize(TRAIN_LABEL_PATH) / (1024 * 1024)

✔ Features saved to data/final/
Train features: /Users/nhatnam/Documents/DM_252/Assignment/data/processed/train_SVM_features.npz
Test features: /Users/nhatnam/Documents/DM_252/Assignment/data/processed/test_SVM_features.npz
Labels: /Users/nhatnam/Documents/DM_252/Assignment/data/processed/train_labels.csv

===== FEATURE SHAPES =====
X_train shape: (2496, 12508)
X_test shape : (596, 12508)

===== LABEL SHAPES =====
y_train length: 2496
